### Setting up

In [ ]:
# Import packages
import uuid
import pandas as pd
import numpy as np
from os import environ
from fabric_api import Auth, Workspace, Dataset, Dataflow, Operations, Database
from dotenv import load_dotenv

# Load .env only if any credential is missing from the environment
_required_vars = ['TENANT_ID', 'CLIENT_ID', 'CLIENT_SECRET']
if not all(environ.get(v) for v in _required_vars):
    load_dotenv('./utils/.env')

# Tenant/app settings
TENANT_ID = environ.get('TENANT_ID', '')
CLIENT_ID = environ.get('CLIENT_ID', '')
CLIENT_SECRET = environ.get('CLIENT_SECRET', '')
FABRIC_SQL_ENDPOINT = environ.get('FABRIC_SQL_ENDPOINT', '')
FABRIC_DATABASE = environ.get('FABRIC_DATABASE', '')

In [ ]:
# Authentication (get bearer token)
auth = Auth(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
pbi_token = auth.get_token('pbi')
fabric_token = auth.get_token('fabric')

# Initializing objects
workspace = Workspace(pbi_token)
dataset = Dataset(pbi_token)
dataflow = Dataflow(pbi_token)
operations = Operations(fabric_token)
db = Database(FABRIC_SQL_ENDPOINT, FABRIC_DATABASE, CLIENT_ID, CLIENT_SECRET)

### Getting workspaces data

In [ ]:
# Get workspaces list
workspaces = workspace.list_workspaces()
workspaces_list = workspaces.get('content', [])

# See content
df_workspaces = pd.DataFrame(workspaces_list)
df_workspaces.sort_values(by='name', inplace=True)
df_workspaces.reset_index(inplace=True, drop=True)
n = len(df_workspaces.index)
print(f'Found {n} workspaces...\nPrinting:')
df_workspaces.head(n)

### List all users with access to a dataset

In [ ]:
#ataset_users = dataset.list_users()
try:
    workspace_to_search = 'Test workspace'
    workspace_to_search_id = df_workspaces['id'][df_workspaces['name']==workspace_to_search].values[0]
except IndexError:
    workspace_to_search_id = ''
print(f'Workspace ID: {workspace_to_search_id}')

dataset_to_search = 'Test dataset'

if workspace_to_search_id != '':
    datasets_list = dataset.list_datasets(workspace_id=workspace_to_search_id)
    df_datasets = pd.DataFrame(datasets_list['content'])
    dataset_to_search_id = df_datasets['id'][df_datasets['name']==dataset_to_search].values[0]
else:
    dataset_to_search_id = ''
print(f'Dataset ID: {dataset_to_search_id}')

if dataset_to_search_id != '':
    dataset_users = dataset.list_users(workspace_id=workspace_to_search_id, dataset_id=dataset_to_search_id)
    df_dataset_users = pd.DataFrame(dataset_users['content'])
    df_dataset_users['workspace'] = workspace_to_search
    df_dataset_users['report'] = dataset_to_search

    df_dataset_users.to_excel(f'./data/users/{dataset_to_search}_users.xlsx', index=False)

### Add user to all workspaces

In [ ]:
users_to_be_added = [    
    {
        'user': 'test@test.com',
        'role': 'Member'
    }
]

for user_to_add in users_to_be_added:

    for workspace_data in workspaces_list:
        workspace_id = workspace_data['id']
        workspace_name = workspace_data['name']
        try:
            response = workspace.add_user(user_principal_name=user_to_add['user'], access_right=user_to_add['role'], workspace_id=workspace_id)
            if response['message'] != 'Success':
                response = workspace.update_user(user_principal_name=user_to_add['user'], access_right=user_to_add['role'], workspace_id=workspace_id)
        except Exception as e:
            print(f'Error:\n\nworkspace_id{workspace_id}\nworkspace_name={workspace_name}\nError message:\n{e}')

### Remove user from all workspaces

In [ ]:
users_to_be_removed = [
    'test@test.com'
]

for user_to_remove in users_to_be_removed:
    for workspace_data in workspaces_list:
        workspace_id = workspace_data['id']
        workspace_name = workspace_data['name']
        try:
            response = workspace.remove_user(user_principal_name=user_to_remove, workspace_id=workspace_id)
            print(f"Removed {user_to_remove} from {workspace_data['name']}")
        except Exception as e:
            print(f'Error: workspace_id{workspace_id} - workspace_name={workspace_name} - Error message: {e}')

### Copy a Dataflow to Another Workspace

In [ ]:
source_workspace_id = '123'
destination_workspace_id = '456'
dataflows_list_raw = dataflow.list_dataflows(workspace_id=source_workspace_id)
dataflows_list = dataflows_list_raw['content']

dataflows_to_copy_list = ['dataflow1', 'dataflow2']
dataflows_to_copy = [d for d in dataflows_list if d.get('name', 'not found') in dataflows_to_copy_list]

In [ ]:
import requests 
for d in dataflows_to_copy:
    dataflow_id = d['objectId']
    dataflow_name = d['name']

    dataflow_content = dataflow.get_dataflow_details(workspace_id=source_workspace_id, dataflow_id=dataflow_id).get('content', '')
    
    if dataflow_content != '':
        
        dataflow_content['entities'][0].pop('partitions', None)
        dataflow_content['pbi:mashup']['allowNativeQueries'] = False

        r = dataflow.create_dataflow(workspace_id=destination_workspace_id, dataflow_content=dataflow_content)
        print(r)